In [1]:
import intake
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

print("Opening PANGEO CMIP6 catalog...")
catalog = intake.open_esm_datastore(
    'https://storage.googleapis.com/cmip6/pangeo-cmip6.json'
)

query = dict(
    experiment_id = ['historical', 'ssp245', 'ssp585'],
    table_id      = 'day',
    variable_id   = ['pr', 'tasmax'],
    source_id     = ['MPI-ESM1-2-HR', 'CMCC-ESM2'],
    member_id     = ['r1i1p1f1'],
)

cat_subset = catalog.search(**query)
print(f"Datasets found: {len(cat_subset.df)}")
print(cat_subset.df[['source_id','experiment_id','variable_id']].to_string())

Opening PANGEO CMIP6 catalog...
Datasets found: 12
        source_id experiment_id variable_id
0   MPI-ESM1-2-HR        ssp245          pr
1   MPI-ESM1-2-HR        ssp245      tasmax
2   MPI-ESM1-2-HR    historical          pr
3   MPI-ESM1-2-HR    historical      tasmax
4   MPI-ESM1-2-HR        ssp585      tasmax
5   MPI-ESM1-2-HR        ssp585          pr
6       CMCC-ESM2    historical      tasmax
7       CMCC-ESM2    historical          pr
8       CMCC-ESM2        ssp585      tasmax
9       CMCC-ESM2        ssp585          pr
10      CMCC-ESM2        ssp245          pr
11      CMCC-ESM2        ssp245      tasmax


In [3]:
# Namibia bounding box
NAMIBIA_BOX = dict(lat=slice(-30, -11), lon=slice(11, 25))

print("Loading datasets and cropping to Namibia...")
print("This may take a few minutes — reading from cloud storage...\n")

dsets = cat_subset.to_dataset_dict(zarr_kwargs={'consolidated': True})

# Cropping each dataset to Namibia and reporting what we have
namibia_dsets = {}
for key, ds in dsets.items():
    try:
        ds_nam = ds.sel(**NAMIBIA_BOX)
        namibia_dsets[key] = ds_nam
        print(f"✓ {key}")
        print(f"  Variables : {list(ds_nam.data_vars)}")
        print(f"  Lat range : {float(ds_nam.lat.min()):.1f} to {float(ds_nam.lat.max()):.1f}")
        print(f"  Lon range : {float(ds_nam.lon.min()):.1f} to {float(ds_nam.lon.max()):.1f}")
        print(f"  Time      : {str(ds_nam.time.values[0])[:10]} to {str(ds_nam.time.values[-1])[:10]}")
        print()
    except Exception as e:
        print(f"✗ {key}: {e}\n")

print(f"Total datasets loaded: {len(namibia_dsets)}")

Loading datasets and cropping to Namibia...
This may take a few minutes — reading from cloud storage...


--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="6" value="6"></progress> 100.00% [6/6 42:53&lt;00:00]</div>

Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information., falling back to GCSFileSystem
Could not determine bucket type for bucket name cmip6: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more 

✓ ScenarioMIP.DKRZ.MPI-ESM1-2-HR.ssp585.day.gn
  Variables : ['tasmax', 'pr']
  Lat range : -29.5 to -11.7
  Lon range : 11.2 to 24.4
  Time      : 2015-01-01 to 2100-12-31

✓ ScenarioMIP.DKRZ.MPI-ESM1-2-HR.ssp245.day.gn
  Variables : ['pr', 'tasmax']
  Lat range : -29.5 to -11.7
  Lon range : 11.2 to 24.4
  Time      : 2015-01-01 to 2100-12-31

✓ ScenarioMIP.CMCC.CMCC-ESM2.ssp585.day.gn
  Variables : ['tasmax', 'pr']
  Lat range : -29.7 to -11.8
  Lon range : 11.2 to 25.0
  Time      : 2015-01-01 to 2100-12-31

✓ ScenarioMIP.CMCC.CMCC-ESM2.ssp245.day.gn
  Variables : ['pr', 'tasmax']
  Lat range : -29.7 to -11.8
  Lon range : 11.2 to 25.0
  Time      : 2015-01-01 to 2100-12-31

✓ CMIP.CMCC.CMCC-ESM2.historical.day.gn
  Variables : ['tasmax', 'pr']
  Lat range : -29.7 to -11.8
  Lon range : 11.2 to 25.0
  Time      : 1850-01-01 to 2014-12-31

✓ CMIP.MPI-M.MPI-ESM1-2-HR.historical.day.gn
  Variables : ['pr', 'tasmax']
  Lat range : -29.5 to -11.7
  Lon range : 11.2 to 24.4
  Time      :

In [ ]:
WINDHOEK_LAT = -22.567
WINDHOEK_LON = 17.100

os.makedirs('../data/analytical', exist_ok=True)

HIST_YEARS = slice('1970', '2014')
FUTR_YEARS = slice('2015', '2060')

print("Extracting Windhoek grid point — point first, then time slice...\n")

for key, ds in namibia_dsets.items():
    try:
        short_key = key.split('.')[-4] + '_' + key.split('.')[-3]
        is_historical = 'historical' in key
        year_slice = HIST_YEARS if is_historical else FUTR_YEARS

        print(f"Processing {short_key}...")

        # Step 1 — extract point FIRST (smallest possible data footprint)
        ds_point = ds.sel(
            lat=WINDHOEK_LAT,
            lon=WINDHOEK_LON,
            method='nearest'
        )

        # Step 2 — then slice time
        ds_point = ds_point.sel(time=year_slice)

        # Step 3 — now load into memory (just one grid cell, ~45 years)
        ds_point = ds_point.compute()

        # Build DataFrame
        dates = ds_point.time.values
        data  = {'DATE': dates}

        if 'pr' in ds_point:
            data['pr_mm_day'] = ds_point['pr'].values * 86400
        if 'tasmax' in ds_point:
            data['tasmax_c'] = ds_point['tasmax'].values - 273.15

        df = pd.DataFrame(data)
        filepath = f'../data/analytical/{short_key}_windhoek.csv'
        df.to_csv(filepath, index=False)

        print(f"  ✓ {len(df)} rows | {df['DATE'].min()} to {df['DATE'].max()}")
        if 'pr_mm_day' in df:
            print(f"  PR mean : {df['pr_mm_day'].mean():.3f} mm/day")
        if 'tasmax_c' in df:
            print(f"  TMAX mean: {df['tasmax_c'].mean():.2f} °C")
        print()

    except Exception as e:
        print(f"  ✗ {short_key}: {e}\n")

print("All files saved to data/analytical/")